# Quick start: FedNCF on MovieLens-100K

This notebook trains FedNCF on MovieLens-100K and reports full-ranking NDCG@10 and HR@10 on the test split. The configuration is the quick start from the README.

FedNCF (Perifanis and Efraimidis, *Knowledge-Based Systems*, 2022) is neural collaborative filtering with one user per client. The server holds the item embeddings and the shared network. Each client holds that user's embeddings.

The steps below install the library, build one dataset, pass it to a server with a client algorithm, a selector, aggregators and an evaluator, train, then score the test split from the best validation round.

## Install

FedRecLab is installed from GitHub. The command also installs NumPy, PyTorch, pandas, tqdm and Cornac. PyTorch is a large download. Run this cell before any import. The notebook does not use the repository working directory, so the same cell works in Colab from a fresh environment.

In [ ]:
%pip install "git+https://github.com/aymenkhs/FedRecLab.git"

## Load the data

`load_cornac_dataset` downloads MovieLens-100K through Cornac and returns an `InteractionDataset`. That object is the only copy of the interactions. User and item ids are remapped there to contiguous integers starting at 0.

The split is stratified by user. With `chrono=True`, each user's earliest interactions stay in training, and the later ones are held out: 10% validation (`val_size=0.1`) and 20% test (`test_size=0.2`). Training reads only the training split. Validation and test rows are read by the evaluator.

`CrossDevicePartitionStrategy` assigns each training user to their own client, so the number of clients is the number of training users. `seed=42` fixes the split.

In [ ]:
from fedreclab.data.load_data import load_cornac_dataset
from fedreclab.data.split import StratifiedSplitStrategy
from fedreclab.data.client_partitioning import CrossDevicePartitionStrategy

dataset = load_cornac_dataset(
    dataset_name="movielens_100k",
    splitting_strategy=StratifiedSplitStrategy(
        include_validation=True, include_test=True,
        val_size=0.1, test_size=0.2, chrono=True, by="user"),
    client_partitioning_strategy=CrossDevicePartitionStrategy(),
    include_validation=True, include_test=True, seed=42,
)

print(
    f"{dataset.num_users} users, {dataset.num_items} items, "
    f"{dataset.num_clients} clients"
)

## Define the method

Three objects, built for this server only.

`FedNCFClientAlgorithm` runs local training. One algorithm object is shared across clients. Each client's user embeddings and optimiser state live in a client state that the server creates. `predictive_factor` is the embedding width and must be the same value on the client and the server. Observed interactions are implicit positives, four unobserved items are sampled as negatives, and the loss is binary cross-entropy. `local_epochs=5` and `lr=5e-4` are the settings used in the README.

`FullRankingEvaluator` scores, for each user, every catalogue item except items from the excluded splits, then averages HR and NDCG at 10. Validation candidates drop that user's training items. Test candidates drop training and validation items. Scoring calls the client algorithm passed in, so the evaluator and the server share that algorithm.

`FedNCF` is the server. Each round, `RandomClientSelection` draws 2% of the clients (`"frac"`). `seed=42` on the server seeds PyTorch and, because this selector was built without its own seed, the client draws as well. The uploaded groups are aggregated separately:

- `item_embeddings` uses `SimpleAvg`. Clients upload the item rows they trained on, and each row is the uniform mean of the clients that sent it.
- `shared_layers` and `output_weights` use `FedAvg`, a mean weighted by how many training interactions the client has.

Item rows are uploaded in the clear, so the server sees which items each selected client updated. The paper's secure-aggregation masking is not part of this run.

`early_stopping` follows validation `ndcg@10` and stops after 10 checks without an improvement. The server keeps the parameters from the best validation round. The test score is computed after `run()`, from that snapshot.

Client algorithms, selectors, aggregators and evaluators are single-use. Build new ones for every server. The dataset may be shared.

In [ ]:
from fedreclab.data.evaluator import FullRankingEvaluator
from fedreclab.federated.client_selection import RandomClientSelection
from fedreclab.federated.aggregation import FedAvg, SimpleAvg
from fedreclab.methods.fedncf.fedncf import FedNCF
from fedreclab.methods.fedncf.fedncf_client import FedNCFClientAlgorithm

client = FedNCFClientAlgorithm(predictive_factor=32, local_epochs=5, lr=5e-4,
                               batch_size=128, neg_samples=4)
evaluator = FullRankingEvaluator(metrics=["ndcg", "hr"], topk=10,
                                 interaction_dataset=dataset, client_algorithm=client)

server = FedNCF(
    predictive_factor=32,
    interaction_dataset=dataset,
    client_algorithm=client,
    num_rounds=100,
    seed=42,
    client_selection_strategy=RandomClientSelection(dataset.num_clients, 0.02, "frac"),
    aggregation_strategies={
        "item_embeddings": SimpleAvg(num_clients=dataset.num_clients),
        "shared_layers": FedAvg(num_clients=dataset.num_clients),
        "output_weights": FedAvg(num_clients=dataset.num_clients),
    },
    evaluator=evaluator,
    early_stopping=True,
    early_stopping_parameters={"metric": "ndcg@10", "direction": "max", "patience": 10},
)

## Train

`run()` trains for up to 100 rounds. An evaluator is attached, so every round ends with a full-catalogue validation. That validation is the slow part of the run. If validation `ndcg@10` stops improving, training ends early.

The parameters used for the test score are the snapshot from the best validation round. Training itself continues from the live parameters until it stops.

In [ ]:
server.run()

## Evaluate

Score the test split with the best validation snapshot: `best_global_parameters`, `best_client_states` and `best_extra_payload`, at round `early_stopping_best_round`.

The result is a dict with keys `ndcg@10` and `hr@10`. Each value is the mean over users who have test interactions.

In [ ]:
print(evaluator.evaluate(
    current_round=server.early_stopping_best_round,
    global_parameters=server.best_global_parameters,
    client_states=server.best_client_states,
    extra_payload=server.best_extra_payload,
    split="test",
))